# Hey-Function-Burmese: FunctionGemma-270M
## End-to-End Fine-Tuning & Multi-Tool Evaluation Pipeline

This notebook provides the complete, reproducible training and evaluation pipeline for **Hey-270M-Alpha**, a lightweight on-device function-calling agent optimized for Burmese, English, and code-switching natural language instructions.

### Key Highlights
- **Base Model:** [`google/functiongemma-270m-it`](https://huggingface.co/google/functiongemma-270m-it)
- **Fine-Tuned Checkpoint:** [`thantzinphyo/Hey-270M-Alpha`](https://huggingface.co/thantzinphyo/Hey-270M-Alpha)
- **Dataset:** [`thantzinphyo/Hey-Function-Burmese`](https://huggingface.co/datasets/thantzinphyo/Hey-Function-Burmese)
- **Training Strategy:** Full-parameter fine-tuning (all 270M weights updated) with response-only loss masking and chunked cross-entropy to prevent VRAM allocation spikes on a 256K vocabulary.
- **Evaluation:** Multi-tool aware exact matching across English, Burmese, and Code-Switching queries with per-tool accuracy breakdown.


In [ ]:
# Step 1: Install Dependencies
!pip install -q -U "transformers>=4.44" "accelerate>=0.33" pyarrow huggingface_hub


In [ ]:
# Step 2: Environment Setup & Hyperparameters
import os
import gc
import re
import math
import json
import time
import random
from functools import partial
from collections import defaultdict, Counter
from typing import List, Dict, Any, Optional

import torch
import torch.nn.functional as F
import pyarrow.parquet as pq
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from huggingface_hub import hf_hub_download

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Optional: Load Hugging Face token from environment if accessing private repositories
HF_TOKEN = os.getenv("HF_TOKEN", None)

# Training Configuration
CONFIG = {
    "model_id": "google/functiongemma-270m-it",
    "data_repo": "thantzinphyo/Hey-Function-Burmese",
    "output_dir": "./functiongemma-270m-alpha",
    "max_seq_len": 2800,       # 2800 tokens guarantees zero sequence truncation
    "epochs": 1,
    "per_device_batch_size": 1,
    "gradient_accumulation_steps": 32,  # Effective batch size: 32
    "learning_rate": 2e-5,
    "warmup_steps": 36,
    "seed": 42,
}

torch.manual_seed(CONFIG["seed"])
random.seed(CONFIG["seed"])

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device: {device_name} | CUDA Available: {torch.cuda.is_available()}")


In [ ]:
# Step 3: Linguistic Register Classifier
# Unicode range definitions for Myanmar script (Burmese)
BURMESE_CODEPOINTS = [(0x1000, 0x109F), (0xA9E0, 0xA9FF), (0xAA60, 0xAA7F)]

def contains_burmese(text: str) -> bool:
    """Checks if text contains any Myanmar (Burmese) script characters."""
    return any(any(start <= ord(char) <= end for start, end in BURMESE_CODEPOINTS) for char in text)

def contains_latin(text: str) -> bool:
    """Checks if text contains Latin / English alphabet characters."""
    return any(65 <= ord(char) <= 90 or 97 <= ord(char) <= 122 for char in text)

def classify_register(text: str) -> str:
    """Classifies user query into English, Burmese, or Code-Switching."""
    has_my = contains_burmese(text)
    has_en = contains_latin(text)
    if has_my and has_en:
        return "Code-Switch"
    elif has_my:
        return "Burmese"
    elif has_en:
        return "English"
    return "Other"

def get_last_user_query(messages: List[Dict[str, Any]]) -> str:
    """Retrieves the latest user query from the message thread."""
    for message in reversed(messages):
        if message.get("role") == "user":
            content = message.get("content", "")
            return content if isinstance(content, str) else str(content)
    return ""


In [ ]:
# Step 4: Dataset Loading & Dynamic Candidate Collation
class FunctionGemmaDataset(torch.utils.data.Dataset):
    """
    Memory-efficient dataset that dynamically includes only the ground-truth target tools
    plus two negative distractor tools per sample. This avoids passing the entire 61-tool
    catalog in every sample, preventing VRAM overflow and sequence truncation.
    """
    def __init__(self, messages_list, global_tools, tokenizer, max_seq_length=2800):
        self.messages_list = messages_list
        self.tools_by_name = {tool["function"]["name"]: tool for tool in global_tools}
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.response_prefix = tokenizer.encode("<start_of_turn>model\n", add_special_tokens=False)

    def __len__(self):
        return len(self.messages_list)

    def __getitem__(self, idx):
        messages = self.messages_list[idx]
        if isinstance(messages, str):
            messages = json.loads(messages)
        elif hasattr(messages, "tolist"):
            messages = messages.tolist()

        # Identify target tool names from ground-truth assistant turns
        target_tools = []
        for message in messages:
            if message.get("tool_calls"):
                for tool_call in message["tool_calls"]:
                    name = tool_call.get("function", {}).get("name")
                    if name:
                        target_tools.append(name)

        # Build candidate set: target tools + 2 distractors
        selected_tools = []
        seen_names = set()
        for name in target_tools:
            if name in self.tools_by_name and name not in seen_names:
                selected_tools.append(self.tools_by_name[name])
                seen_names.add(name)

        distractors = [tool for name, tool in self.tools_by_name.items() if name not in seen_names]
        selected_tools.extend(distractors[:2])

        # Format prompt with FunctionGemma chat template
        text = self.tokenizer.apply_chat_template(messages, tools=selected_tools, tokenize=False)
        input_ids = self.tokenizer(
            text, truncation=True, max_length=self.max_seq_length, add_special_tokens=False
        )["input_ids"]

        # Response-only loss masking: mask system and user tokens with -100
        labels = list(input_ids)
        prefix_len = len(self.response_prefix)
        match_index = -1
        for i in range(len(input_ids) - prefix_len + 1):
            if input_ids[i:i + prefix_len] == self.response_prefix:
                match_index = i + prefix_len
                break

        if match_index != -1:
            labels[:match_index] = [-100] * match_index
        else:
            labels = [-100] * len(labels)

        return {"input_ids": input_ids, "labels": labels}

def causal_collate_fn(batch, pad_id=0):
    """Dynamically pads batches to the longest sequence in the batch."""
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, labels, attention_mask = [], [], []
    for item in batch:
        inp, lbl = item["input_ids"], item["labels"]
        padding_length = max_len - len(inp)
        input_ids.append(inp + [pad_id] * padding_length)
        labels.append(lbl + [-100] * padding_length)
        attention_mask.append([1] * len(inp) + [0] * padding_length)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


In [ ]:
# Step 5: Chunked Cross-Entropy Loss & Clean Trainer
def chunked_cross_entropy(logits, labels, chunk_size=256, ignore_index=-100, reduction="mean"):
    """
    Zero-copy chunked cross-entropy. FunctionGemma uses a 256,000 token vocabulary;
    computing cross-entropy over full unchunked logits would allocate >8 GB VRAM at once.
    Chunking sequence positions keeps peak memory under 5 GB.
    """
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    flat_logits = shift_logits.reshape(-1, shift_logits.size(-1))
    flat_labels = shift_labels.reshape(-1)
    total_active_tokens = (flat_labels != ignore_index).sum()
    if total_active_tokens == 0:
        return flat_logits.sum() * 0.0

    accumulated_loss = 0.0
    for i in range(0, flat_labels.size(0), chunk_size):
        chunk_labels = flat_labels[i:i + chunk_size]
        if (chunk_labels != ignore_index).any():
            accumulated_loss = accumulated_loss + F.cross_entropy(
                flat_logits[i:i + chunk_size].float(),
                chunk_labels,
                ignore_index=ignore_index,
                reduction="sum",
            )
    return accumulated_loss if reduction == "sum" else accumulated_loss / total_active_tokens

class CleanCausalTrainer(Trainer):
    """Custom Trainer utilizing chunked cross-entropy loss computation."""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = chunked_cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

class TrainingProgressCallback(TrainerCallback):
    """Real-time training monitor tracking loss, throughput, and GPU memory."""
    def __init__(self, total_steps):
        self.total_steps = total_steps
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("=" * 75)
        print(f"TRAINING INITIATED | Planned Steps: {self.total_steps:,}")
        print("=" * 75, flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs: return
        step = state.global_step
        loss = logs["loss"]
        lr = logs.get("learning_rate", 0.0)
        elapsed = time.time() - (self.start_time or time.time())
        sec_per_step = elapsed / max(step, 1)
        eta_minutes = int((self.total_steps - step) * sec_per_step // 60)
        vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
        print(
            f"[Step {step:4d}/{self.total_steps}] Loss: {loss:.4f} | LR: {lr:.2e} | "
            f"{sec_per_step:.1f}s/step (ETA {eta_minutes}m) | Peak VRAM: {vram_gb:.2f} GB",
            flush=True,
        )


In [ ]:
# Step 6: Full Fine-Tuning Execution
def train_model():
    print("[1/3] Downloading Hey-Function-Burmese dataset from Hugging Face Hub...")
    data_path = hf_hub_download(CONFIG["data_repo"], "train.parquet", repo_type="dataset", token=HF_TOKEN)
    table = pq.read_table(data_path, columns=["messages", "tools", "metadata"])
    global_tools = table.column("tools")[0].as_py()
    messages_column = table.column("messages").to_pylist()
    metadata_column = table.column("metadata").to_pylist()

    train_messages = [msg for msg, meta in zip(messages_column, metadata_column) if meta != "eval"]
    eval_messages = [msg for msg, meta in zip(messages_column, metadata_column) if meta == "eval"]
    print(f"  → Dataset Loaded: {len(train_messages):,} training samples, {len(eval_messages):,} evaluation samples")

    print(f"[2/3] Initializing Tokenizer and Model: {CONFIG['model_id']}...")
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"], token=HF_TOKEN)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_dataset = FunctionGemmaDataset(train_messages, global_tools, tokenizer, CONFIG["max_seq_len"])
    eval_subset = random.Random(CONFIG["seed"]).sample(eval_messages, min(500, len(eval_messages)))
    eval_dataset = FunctionGemmaDataset(eval_subset, global_tools, tokenizer, CONFIG["max_seq_len"])

    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"], token=HF_TOKEN, torch_dtype=torch.float32, attn_implementation="eager",
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    total_steps = math.ceil(len(train_dataset) / (CONFIG["per_device_batch_size"] * CONFIG["gradient_accumulation_steps"])) * CONFIG["epochs"]
    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        per_device_train_batch_size=CONFIG["per_device_batch_size"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        num_train_epochs=CONFIG["epochs"],
        warmup_steps=CONFIG["warmup_steps"],
        logging_steps=5,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        report_to="none",
    )

    trainer = CleanCausalTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=partial(causal_collate_fn, pad_id=tokenizer.pad_token_id),
        callbacks=[TrainingProgressCallback(total_steps)],
    )

    print("[3/3] Executing Fine-Tuning...")
    trainer.train()
    trainer.save_model(CONFIG["output_dir"])
    tokenizer.save_pretrained(CONFIG["output_dir"])
    print(f"SUCCESS: Fine-tuned artifacts successfully saved to '{CONFIG['output_dir']}'!")

# Uncomment to run training:
# train_model()


## Evaluation Benchmark

This section evaluates the fine-tuned model across held-out evaluation samples in **English**, **Burmese**, and **Code-Switching**.

### Metrics Computed
1. **Function Accuracy (Fn Acc):** Percentage of queries where all predicted tool names match ground truth.
2. **Argument Accuracy (Arg Acc):** Percentage of queries where both tool names and all extracted arguments match ground truth.
3. **Exact Match (EM):** Strict end-to-end match across functions and arguments.
4. **Per-Tool Accuracy:** Tool invocation success rate broken down across all supported tools.


In [ ]:
# Step 7: Multi-Tool Function Calling Evaluation
def parse_function_arguments(args_string: str) -> Dict[str, Any]:
    """Parses key-value pairs formatted by FunctionGemma syntax."""
    arguments = {}
    idx, length = 0, len(args_string)
    while idx < length:
        while idx < length and args_string[idx] in ', \n\t':
            idx += 1
        if idx >= length: break
        colon_idx = args_string.find(':', idx)
        if colon_idx < 0: break
        key = args_string[idx:colon_idx].strip()
        idx = colon_idx + 1

        if args_string.startswith('<escape>', idx):
            escape_end = args_string.find('<escape>', idx + 8)
            if escape_end < 0:
                arguments[key] = args_string[idx + 8:]
                idx = length
            else:
                arguments[key] = args_string[idx + 8:escape_end]
                idx = escape_end + 8
        else:
            comma_idx = args_string.find(',', idx)
            if comma_idx < 0:
                comma_idx = length
            raw_value = args_string[idx:comma_idx].strip()
            idx = comma_idx
            if raw_value in ('None', ''):
                arguments[key] = None
            elif raw_value.lower() == 'true':
                arguments[key] = True
            elif raw_value.lower() == 'false':
                arguments[key] = False
            else:
                try:
                    arguments[key] = float(raw_value) if '.' in raw_value else int(raw_value)
                except Exception:
                    arguments[key] = raw_value
    return arguments

def extract_function_calls(text: str) -> List[Dict[str, Any]]:
    """Extracts all function calls matching the FunctionGemma token pattern."""
    if not text or not text.strip():
        return []
    pattern = r'<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>'
    return [
        {"name": match.group(1), "arguments": parse_function_arguments(match.group(2))}
        for match in re.finditer(pattern, text, re.DOTALL)
    ]

def clean_arguments(arguments: Any) -> Dict[str, Any]:
    """Removes non-called null parameters from comparison."""
    return {k: v for k, v in arguments.items() if v is not None} if isinstance(arguments, dict) else {}

def value_equals(val1: Any, val2: Any) -> bool:
    """Compares two values with case-insensitivity and float tolerance."""
    s1, s2 = str(val1).strip().lower(), str(val2).strip().lower()
    if s1 == s2: return True
    try: return float(s1) == float(s2)
    except Exception: return False

def compare_arguments(ground_truth: Dict[str, Any], predicted: Dict[str, Any]) -> bool:
    """Checks if all ground truth arguments match prediction arguments."""
    gt_clean = clean_arguments(ground_truth)
    pred_clean = clean_arguments(predicted)
    if set(gt_clean.keys()) != set(pred_clean.keys()):
        return False
    return all(value_equals(pred_clean[key], gt_clean[key]) for key in gt_clean)

def run_benchmark(model_path="thantzinphyo/Hey-270M-Alpha", sample_count=200):
    """Runs multi-tool aware benchmark across linguistic registers."""
    print(f"[Evaluation] Loading model: {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, token=HF_TOKEN, torch_dtype=torch.float32, attn_implementation="eager",
    ).cuda().eval()

    data_path = hf_hub_download(CONFIG["data_repo"], "train.parquet", repo_type="dataset", token=HF_TOKEN)
    table = pq.read_table(data_path, columns=["messages", "tools", "metadata"])
    global_tools = table.column("tools")[0].as_py()
    tools_by_name = {tool["function"]["name"]: tool for tool in global_tools}
    messages_column = table.column("messages").to_pylist()
    metadata_column = table.column("metadata").to_pylist()
    eval_messages = [msg for msg, meta in zip(messages_column, metadata_column) if meta == "eval"]

    parsed_samples = [
        json.loads(m) if isinstance(m, str) else (m.tolist() if hasattr(m, 'tolist') else m)
        for m in eval_messages
    ]
    registers = [classify_register(get_last_user_query(m)) for m in parsed_samples]

    samples_by_register = defaultdict(list)
    for idx, reg in enumerate(registers):
        samples_by_register[reg].append(idx)

    selected_indices = []
    for reg, indices in samples_by_register.items():
        quota = max(1, round(sample_count * len(indices) / len(registers)))
        selected_indices.extend(random.sample(indices, min(quota, len(indices))))
    selected_indices = selected_indices[:sample_count]
    random.shuffle(selected_indices)
    print(f"[Evaluation] Running evaluation on {len(selected_indices)} held-out samples...")

    register_results = defaultdict(lambda: {"total": 0, "exact": 0, "wrong_args": 0, "wrong_tool": 0})
    overall_results = {"total": 0, "exact": 0, "wrong_args": 0, "wrong_tool": 0}
    tool_performance = defaultdict(lambda: {"total": 0, "correct": 0})

    for sample_idx in selected_indices:
        messages = parsed_samples[sample_idx]
        register = registers[sample_idx]

        gt_calls = []
        for message in messages:
            if message.get("tool_calls"):
                for tc in message["tool_calls"]:
                    gt_calls.append(tc.get("function", {}))
        if not gt_calls: continue
        gt_names = [call["name"] for call in gt_calls]

        seen = set()
        sample_tools = []
        for name in gt_names:
            if name in tools_by_name and name not in seen:
                sample_tools.append(tools_by_name[name])
                seen.add(name)
        sample_tools.extend([tool for name, tool in tools_by_name.items() if name not in seen][:2])

        prompt_messages = [m for m in messages if m.get("role") != "assistant"]
        prompt = tokenizer.apply_chat_template(prompt_messages, tools=sample_tools, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2800).to("cuda")

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        generated_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        predicted_calls = extract_function_calls(generated_text)

        if not predicted_calls:
            category = "wrong_tool"
        else:
            pred_names = [call["name"] for call in predicted_calls]
            if set(pred_names) != set(gt_names):
                category = "wrong_tool"
            else:
                is_exact = all(
                    next((compare_arguments(gt["arguments"], pred["arguments"]) for pred in predicted_calls if pred["name"] == gt["name"]), False)
                    for gt in gt_calls
                )
                category = "exact" if is_exact else "wrong_args"

        register_results[register]["total"] += 1
        overall_results["total"] += 1
        register_results[register][category] += 1
        overall_results[category] += 1

        for gt in gt_calls:
            tool_name = gt["name"]
            tool_performance[tool_name]["total"] += 1
            matching_call = next((pred for pred in predicted_calls if pred["name"] == tool_name), None)
            if matching_call and compare_arguments(gt["arguments"], matching_call["arguments"]):
                tool_performance[tool_name]["correct"] += 1

    def percentage(num, den): return (num / den * 100) if den else 0.0

    print("\n" + "=" * 70)
    print(f"Evaluation Benchmark (Held-out Set: {overall_results['total']} samples)")
    print("=" * 70)
    print(f"{'Register':<14}{'Samples':>8}   {'Fn Acc':>8}   {'Arg Acc':>9}   {'EM':>7}")
    print("-" * 70)
    for reg in ["English", "Burmese", "Code-Switch"]:
        res = register_results[reg]
        if res["total"] == 0: continue
        fn_acc = res["exact"] + res["wrong_args"]
        print(
            f"{reg:<14}{res['total']:>8}   "
            f"{percentage(fn_acc, res['total']):>7.1f}%   "
            f"{percentage(res['exact'], res['total']):>8.1f}%   "
            f"{percentage(res['exact'], res['total']):>6.1f}%"
        )
    print("-" * 70)
    total_fn = overall_results["exact"] + overall_results["wrong_args"]
    print(
        f"{'Overall':<14}{overall_results['total']:>8}   "
        f"{percentage(total_fn, overall_results['total']):>7.1f}%   "
        f"{percentage(overall_results['exact'], overall_results['total']):>8.1f}%   "
        f"{percentage(overall_results['exact'], overall_results['total']):>6.1f}%\n"
    )

    print("Per-Tool Accuracy (min 3 samples):")
    tool_stats = [
        (name, data["correct"], data["total"], percentage(data["correct"], data["total"]))
        for name, data in tool_performance.items() if data["total"] >= 3
    ]
    tool_stats.sort(key=lambda item: item[3], reverse=True)
    print(f"{'Tool Name':<32}{'Correct':>9}{'Total':>7}{'Acc':>8}")
    print("-" * 58)
    for name, correct, total, acc in tool_stats:
        print(f"{name:<32}{correct:>9}{total:>7}{acc:>7.1f}%")
    print("=" * 70)

# Run evaluation against published checkpoint
run_benchmark("thantzinphyo/Hey-270M-Alpha")
